In [1]:
import sqlite3, pandas as pd
con = sqlite3.connect("practice.db")
con.executescript(open("practice_setup.sql").read())
def run(q): return pd.read_sql(q, con)

In [2]:
"""-- 1. Filter + sort + limit  [Service / screening]
-- Return name and salary of Engineering-dept (dept_id 1) employees
-- earning more than 70000, highest salary first.
"""
query1 = """SELECT name,salary FROM employees where dept_id=1 and salary > 70000 order by salary desc;"""
run(query1)

,name,salary
0,Arjun,95000
1,Divya,81000
2,Priya,72000


In [4]:
"""-- 2. GROUP BY + HAVING + sort  [Service / aggregation]
-- Return dept_id and headcount, ONLY for departments with more than 2
-- employees, sorted by headcount descending."""
query2 = """SELECT dept_id,COUNT(name) as 'count' from employees group by dept_id having count > 2;"""
run(query2)

,dept_id,count
0,1,5
1,2,4


In [5]:
"""-- 3. LEFT JOIN + IS NULL  [AI screen / joins]
-- Return the name of every employee whose department does NOT exist
-- in the departments table. (Think LEFT JOIN, then filter.)"""
query3 = """SELECT e.name, from employees e LEFT JOIN departments d on d.dept_id = e.dept_id"""
run(query3)

,name
0,Arjun
1,Priya
2,Karan
3,Divya
4,Rohit
5,Sneha
6,Vikram
7,Anita
8,Manoj
9,Kavya


In [6]:
"""-- 4. 3-table join + filter  [AI screen / joins]
-- Return employee name, project_name, and hours
-- for assignments with more than 150 hours, highest hours first."""
query4 = """SELECT e.name,p.project_name,a.hours from assignments a 
join employees e on e.emp_id = a.emp_id
join projects p on p.project_id = a.project_id
where a.hours>150
order by a.hours desc"""
run(query4)

,name,project_name,hours
0,Divya,Mobile App,210
1,Priya,Payment Gateway,200
2,Deepa,Q4 Campaign,180
3,Kavya,Audit System,170
4,Karan,Payment Gateway,160


In [7]:
"""-- 5. Subquery  [AI screen / subquery]
-- Return name and salary of everyone who earns more than the
-- AVERAGE salary of the Sales department (dept_id 2)."""
query5 = """Select name,salary from employees where salary > (select AVG(salary) from employees where dept_id=12)"""
run(query5)

,name,salary


In [11]:
"""-- 6. GROUP BY with JOIN  [AI screen / aggregation]
-- Return dept_name and the average salary in that department,
-- sorted highest average first. Use the department NAME, not the id."""
query6 = """SELECT d.dept_name,AVG(e.salary) as 'avg_salary' from departments d JOIN employees e on e.dept_id = d.dept_id
group by e.dept_id
order by avg_salary desc"""
run(query6)

,dept_name,avg_salary
0,Engineering,77200.0
1,Finance,70000.0
2,Sales,66250.0
3,Marketing,59000.0
4,HR,48500.0


In [13]:
"""-- 7. Window: rank within group + filter  [AI screen / window]
-- Rank employees by salary within each department (highest = 1),
-- then return ONLY the top earner in each department:
-- columns name, dept_id, salary. (wrap the window query and filter rank = 1)"""
query7="""SELECT * from (SELECT name,dept_id,salary , RANK() over(partition by dept_id order by salary desc) as 'rnk' from employees) as ranked where ranked.rnk=1"""
run(query7)

,name,dept_id,salary,rnk
0,Arjun,1,95000,1
1,Sneha,2,88000,1
2,Anita,3,50000,1
3,Kavya,4,76000,1
4,Deepa,5,59000,1
5,Nisha,6,52000,1


In [16]:
"""-- 8. Running total  [AI screen / window]
-- For emp_id 6, return sale_date, amount, and a running total of amount
-- ordered by sale_date.
"""
query8="""SELECT sale_date,amount, SUM(amount) OVER(order by sale_date asc) as "running_total" from sales where emp_id=6"""
run(query8)

,sale_date,amount,running_total
0,2024-01-12,25000,25000
1,2024-02-15,30000,55000
2,2024-03-18,22000,77000


In [19]:
"""-- 9. COUNT with LEFT JOIN (the zero case)  [AI screen / joins]
-- Return every dept_name and how many employees it has.
-- Departments with ZERO employees must still appear with a count of 0."""
query9 = """Select d.dept_name, count(name) as 'count' from employees e 
LEFT join departments d on d.dept_id=e.dept_id
group by e.dept_id"""
run(query9)

,dept_name,count
0,Engineering,5
1,Sales,4
2,HR,2
3,Finance,2
4,Marketing,1
5,None,1
